# 01 · TSFM change-point scores (Colab, GPU)

**Project:** Do Zero-Shot Time-Series Foundation Models Detect Change Points? (P1)
**Written:** 2026-10-11 · **Runtime:** free T4 GPU, estimated 30–90 min in total. The run is resumable.

**What it does.** For every series in TCPD (36) and SKAB (34) it writes per-time-step scores for three zero-shot detectors. No training or fine-tuning is done.

| Output folder | Model | Detector |
|---|---|---|
| `chronos2_residual` | Chronos-2 (`amazon/chronos-2`) | one-step forecast surprise, `abs(x - q50) / (q90 - q10)`, for each context length L in {32, 64, 128, 256} |
| `timesfm25_residual` | TimesFM-2.5 (`google/timesfm-2.5-200m-pytorch`) | the same residual detector (channels are forecast one at a time) |
| `chronos2_embedding` | Chronos-2 | split-window embedding distance, for half-window w in {16, 32, 64} |

**Inputs.** Upload `colab_input.zip` to Google Drive at `MyDrive/p1_tsfm/colab_input.zip`. It contains `configs/`, `src/`, `data/processed/` and `requirements-colab.txt`.

**Outputs** go to `MyDrive/p1_tsfm/`:
- `tsfm_raw/<detector>/<series>.npz`, one score array per parameter value plus runtimes
- `tsfm_raw/manifest.json`
- `tsfm_raw.zip`

**How to run.**
1. Choose *Runtime → Change runtime type → T4 GPU*.
2. Choose *Runtime → Run all*.
3. If the session disconnects, run all again. Finished series are skipped.

In [ ]:
# Setup: mount Drive, unpack inputs, install the pinned packages, check the GPU
import os, sys, zipfile, importlib.metadata as md
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/p1_tsfm'
WORK = '/content/work'
assert os.path.exists(f'{DRIVE}/colab_input.zip'), f'Upload colab_input.zip to {DRIVE}/ first'
if not os.path.exists(f'{WORK}/configs'):
    zipfile.ZipFile(f'{DRIVE}/colab_input.zip').extractall(WORK)

!pip install -q -r /content/work/requirements-colab.txt
sys.path.insert(0, f'{WORK}/src')

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
PACKAGES = ['chronos-forecasting', 'timesfm', 'torch', 'transformers', 'numpy']
print('GPU:', torch.cuda.get_device_name(0))
print({p: md.version(p) for p in PACKAGES}, '| python', sys.version.split()[0])

In [ ]:
# Config: the same configs/*.yaml the local code uses; no parameters are hard-coded here
import json, time, shutil, datetime
import numpy as np
from cpd import load_config
from cpd.data.loaders import load_processed, zscore
from cpd.detectors.scores import embedding_scores, residual_scores

MODELS, DET = load_config('models'), load_config('detectors')
R, E, CC = DET['residual'], DET['embedding'], DET['colab']
torch.manual_seed(DET['seed']); np.random.seed(DET['seed'])
data = load_processed(f'{WORK}/data/processed')
OUT = f'{DRIVE}/tsfm_raw'
os.makedirs(OUT, exist_ok=True)
print(len(data), 'series:', sum(s['dataset'] == 'tcpd' for s in data.values()), 'TCPD +',
      sum(s['dataset'] == 'skab' for s in data.values()), 'SKAB')

In [ ]:
# Load both models at the revisions pinned in configs/models.yaml
import timesfm
from chronos import Chronos2Pipeline

c, t = MODELS['chronos-2'], MODELS['timesfm-2.5']
chronos = Chronos2Pipeline.from_pretrained(c['hf_id'], revision=c['revision'], device_map='cuda')
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(t['hf_id'], revision=t['revision'], torch_compile=False)
tfm.compile(timesfm.ForecastConfig(
    max_context=CC['timesfm_max_context'], max_horizon=CC['timesfm_max_horizon'],
    per_core_batch_size=CC['timesfm_batch_size'], normalize_inputs=True,
    use_continuous_quantile_head=True, force_flip_invariance=True,
    infer_is_positive=False,          # inputs are z-scored, so they can be negative
    fix_quantile_crossing=True))
print('models loaded')

In [ ]:
# Model adapters: each turns a list of (C, l) arrays into what cpd.detectors.scores expects
Q = R['quantiles']                                  # [0.1, 0.5, 0.9]
TFM_Q = [round(q * 10) for q in Q]                  # TimesFM quantile axis: 0 = mean, k = q0.k -> [1, 5, 9]
PATCH = chronos.model.chronos_config.input_patch_size
N_SPECIAL = 1 + int(chronos.model.chronos_config.use_reg_token)   # [REG] token + masked output patch

def chronos_forecast(contexts):
    q, _ = chronos.predict_quantiles(contexts, prediction_length=1, quantile_levels=Q,
                                     batch_size=CC['chronos_batch_size'])
    q = torch.stack([x[:, 0, :] for x in q]).float().cpu().numpy()      # (n, C, 3)
    return q[..., 0], q[..., 1], q[..., 2]

def timesfm_forecast(contexts):
    n, C = len(contexts), contexts[0].shape[0]
    flat = [ctx[ch] for ctx in contexts for ch in range(C)]             # univariate model: one series per channel
    _, qf = tfm.forecast(horizon=1, inputs=flat)                         # (n*C, 1, 10)
    qf = qf[:, 0, :].reshape(n, C, -1)
    return tuple(qf[..., k] for k in TFM_Q)

def chronos_embed(windows):
    n_patch = windows[0].shape[1] // PATCH
    emb, _ = chronos.embed(windows, batch_size=CC['chronos_batch_size'], context_length=windows[0].shape[1])
    # keep the context-patch tokens (time order), then average over channels -> (P, d)
    return np.stack([e[:, -N_SPECIAL - n_patch:-N_SPECIAL].float().mean(0).numpy() for e in emb])

# Smoke test on real series before the long loop, so problems show up in the first minute
z = zscore(data['tcpd/nile']['values'])
with torch.inference_mode():
    for name, fn in [('chronos', chronos_forecast), ('timesfm', timesfm_forecast)]:
        s = residual_scores(z, fn, 64, R['min_context'], R['eps'])
        print(f'{name:8s} residual on nile: argmax t={int(np.argmax(s))}, max score={s.max():.2f}')
    s = embedding_scores(z, chronos_embed, 16)
    print(f'chronos  embedding on nile: argmax t={int(np.argmax(s))}, max={s.max():.3f}')
z = zscore(data['skab/valve1_0']['values'])
with torch.inference_mode():
    for fn in (timesfm_forecast, chronos_forecast):
        q10, q50, q90 = fn([z[:50].T.astype(np.float32)])
        assert q50.shape == (1, 8) and (q90 >= q10).all()
print('smoke test passed (for reference, nile annotators mark the change at t=28)')

In [ ]:
# Main loop: resumable, one .npz per (detector, series); finished files are skipped
JOBS = [('chronos2_residual', 'residual', chronos_forecast),
        ('timesfm25_residual', 'residual', timesfm_forecast),
        ('chronos2_embedding', 'embedding', chronos_embed)]
if not os.path.exists(f'{OUT}/started.txt'):
    open(f'{OUT}/started.txt', 'w').write(datetime.datetime.now().isoformat())

for job, kind, fn in JOBS:
    os.makedirs(f'{OUT}/{job}', exist_ok=True)
    params = R['context_lengths'] if kind == 'residual' else E['half_windows']
    for sid, s in data.items():
        path = f"{OUT}/{job}/{sid.replace('/', '__')}.npz"
        if os.path.exists(path):
            continue
        z = zscore(s['values'])
        arrays = {}
        for p in params:
            t0 = time.perf_counter()
            with torch.inference_mode():
                sc = (residual_scores(z, fn, p, R['min_context'], R['eps']) if kind == 'residual'
                      else embedding_scores(z, fn, p))
            torch.cuda.synchronize()
            arrays[f'p{p}'], arrays[f'runtime_p{p}'] = sc, time.perf_counter() - t0
        np.savez_compressed(path + '.tmp.npz', **arrays)
        os.replace(path + '.tmp.npz', path)       # never leave a half-written file
        print(f"{job:20s} {sid:26s} T={s['n_obs']:5d}  " +
              ' '.join(f"p{p}:{arrays[f'runtime_p{p}']:.1f}s" for p in params), flush=True)

In [ ]:
# Manifest, completeness check and zip for download
expected = len(data) * len(JOBS)
written = sum(f.endswith('.npz') and not f.endswith('.tmp.npz')
              for job, _, _ in JOBS for f in os.listdir(f'{OUT}/{job}'))
manifest = dict(
    notebook='01_tsfm_scores.ipynb',
    models={k: f"{v['hf_id']}@{v['revision']}" for k, v in MODELS.items()},
    packages={p: md.version(p) for p in PACKAGES},
    python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), seed=DET['seed'],
    series_expected=expected, series_written=written,
    started=open(f'{OUT}/started.txt').read(), finished=datetime.datetime.now().isoformat())
json.dump(manifest, open(f'{OUT}/manifest.json', 'w'), indent=1)
print(json.dumps(manifest, indent=1))
zip_path = shutil.make_archive(f'{DRIVE}/tsfm_raw', 'zip', OUT)
print(f'\n{written}/{expected} files written. Download: {zip_path}')
assert written == expected, 'Some series are missing: run all cells again to resume'